### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat graphs as adjacency lists for BFS/DFS. Seniors see the adjacency matrix as a **learnable linear operator** — multiplying by it is equivalent to aggregating neighbor features, which IS the convolution on graphs. Understanding this duality between graph topology and linear algebra is what separates a GNN practitioner from someone who just calls `model.fit()`.

**Mental Model:** Think of each node as a person in a room. In each "round" (layer), every person:
1. **Collects** messages from their neighbors (AGGREGATE).
2. **Combines** those messages with their own knowledge (UPDATE).
3. **Broadcasts** their updated knowledge for the next round.

After $K$ rounds, each person knows about everyone up to $K$ hops away — without ever leaving their seat.

**Common Junior Pitfall:** Stacking too many GNN layers. Unlike CNNs where depth helps, GNNs suffer from **over-smoothing**: after ~6 layers, all node representations converge to the same vector (every node "knows" the entire graph and looks identical). Seniors use 2-3 layers with skip connections.

---


<a id="1"></a>
## 1. Graph Theory Fundamentals for Neural Networks

A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ consists of:
- **Nodes (Vertices)** $\mathcal{V} = \{v_1, v_2, \ldots, v_N\}$: Entities (users, atoms, transactions).
- **Edges** $\mathcal{E} \subseteq \mathcal{V} \times \mathcal{V}$: Relationships between entities.
- **Node Feature Matrix** $X \in \mathbb{R}^{N \times F}$: Each node has an $F$-dimensional feature vector.

### The Adjacency Matrix $A$
For $N$ nodes, we encode all edges in a binary matrix $A \in \{0, 1\}^{N \times N}$:
$$A_{ij} = \begin{cases} 1 & \text{if } (v_i, v_j) \in \mathcal{E} \\ 0 & \text{otherwise} \end{cases}$$

For undirected graphs, $A$ is symmetric: $A = A^T$.

### The Degree Matrix $D$
A diagonal matrix where $D_{ii} = \sum_j A_{ij}$ (i.e., the number of neighbors of node $i$).

### The Graph Laplacian $L$
$$L = D - A$$

The Laplacian is to graphs what the second derivative is to continuous functions. Its eigenvalues encode the graph's "frequency spectrum" — this is the foundation of **spectral graph theory** and the motivation for GCNs.

### 🎓 Socratic Deep Check
> *"If you multiply the adjacency matrix $A$ by the feature matrix $X$, i.e. compute $AX$, what does each row of the result represent? Why is this single matrix multiplication the conceptual core of ALL graph neural networks?"*

<details><summary>👉 Answer</summary>
Row $i$ of $AX$ is the SUM of all feature vectors of node $i$'s neighbors: $(AX)_i = \sum_{j \in \mathcal{N}(i)} x_j$. This single multiplication IS neighborhood aggregation. Every GNN architecture is fundamentally a variation of applying $AX$ with different normalization, weighting, and nonlinearities.
</details>


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Graph Theory Fundamentals for Neural Networks](#1-graph-theory-fundamentals-for-neural-networks)
  * [The Adjacency Matrix $A$](#the-adjacency-matrix-a)
  * [The Degree Matrix $D$](#the-degree-matrix-d)
  * [The Graph Laplacian $L$](#the-graph-laplacian-l)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
* [2. The Message Passing Neural Network (MPNN) Framework](#2-the-message-passing-neural-network-mpnn-framework)
  * [Step 1: Message Construction](#step-1-message-construction)
  * [Step 2: Aggregation](#step-2-aggregation)
  * [Step 3: Update](#step-3-update)
  * [The Receptive Field](#the-receptive-field)
* [3. Graph Convolutional Networks (GCN) — Kipf & Welling, 2017](#3-graph-convolutional-networks-gcn-kipf-welling-2017)
  * [Motivation: From Spectral Theory to a Simple Formula](#motivation-from-spectral-theory-to-a-simple-formula)
  * [The GCN Layer Rule](#the-gcn-layer-rule)
  * [Why Symmetric Normalization?](#why-symmetric-normalization)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [Limitation: GCN is Transductive](#limitation-gcn-is-transductive)
* [4. GraphSAGE — Inductive Representation Learning (Hamilton et al., 2017)](#4-graphsage-inductive-representation-learning-hamilton-et-al-2017)
  * [The Problem with GCN](#the-problem-with-gcn)
  * [The GraphSAGE Innovation](#the-graphsage-innovation)
  * [The Algorithm](#the-algorithm)
  * [Key Differences from GCN](#key-differences-from-gcn)
  * [Why CONCAT Instead of SUM?](#why-concat-instead-of-sum)
* [5. Graph Attention Networks (GAT) — Veličković et al., 2018](#5-graph-attention-networks-gat-veličković-et-al-2018)
  * [The Problem with Fixed Aggregation](#the-problem-with-fixed-aggregation)
  * [The GAT Innovation: Learned Attention Coefficients](#the-gat-innovation-learned-attention-coefficients)
  * [The Mathematics](#the-mathematics)
  * [Multi-Head Attention](#multi-head-attention)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
* [6. Application — Molecular Property Prediction (Node Classification)](#6-application-molecular-property-prediction-node-classification)
  * [The Problem](#the-problem)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: What does multiplying $AX$ compute, and why is it the foundation of all GNNs?](#q1-what-does-multiplying-ax-compute-and-why-is-it-the-foundation-of-all-gnns)
  * [Q2: Why does GCN use symmetric normalization $\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$ instead of simple row normalization $D^{-1}A$?](#q2-why-does-gcn-use-symmetric-normalization-tilded-12-tildea-tilded-12-instead-of-simple-row-normalization-d-1a)
  * [Q3: Why is GraphSAGE called "inductive" while GCN is "transductive"?](#q3-why-is-graphsage-called-inductive-while-gcn-is-transductive)
  * [Q4: How does GAT determine the importance of each neighbor?](#q4-how-does-gat-determine-the-importance-of-each-neighbor)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- Build a small graph from scratch ---
# 5-node graph: 0-1, 0-2, 1-2, 2-3, 3-4
edge_index = torch.tensor([[0,1], [0,2], [1,0], [1,2], [2,0], [2,1], [2,3], [3,2], [3,4], [4,3]], dtype=torch.long).t()

num_nodes = 5

# --- Adjacency Matrix ---
A = torch.zeros(num_nodes, num_nodes)
A[edge_index[0], edge_index[1]] = 1.0
print("Adjacency Matrix A:")
print(A)

# --- Degree Matrix ---
D = torch.diag(A.sum(dim=1))
print("\nDegree Matrix D:")
print(D)

# --- Graph Laplacian ---
L = D - A
print("\nGraph Laplacian L = D - A:")
print(L)

# --- The AX trick: Neighborhood Aggregation ---
# Each node has a 3-dim feature vector
X = torch.tensor([[1.0, 0.0, 0.0],   # Node 0
                   [0.0, 1.0, 0.0],   # Node 1
                   [0.0, 0.0, 1.0],   # Node 2
                   [1.0, 1.0, 0.0],   # Node 3
                   [0.0, 1.0, 1.0]])  # Node 4

AX = A @ X
print("\nAX (sum of neighbor features):")
print(AX)
print(f"\nNode 2 neighbors are {{0, 1, 3}}: sum = {X[0] + X[1] + X[3]}")
print(f"Row 2 of AX:                          {AX[2]}")
print("✅ They match — AX IS neighborhood aggregation.")

"""
What just happened?
We proved that a single matrix multiplication A @ X computes the sum of all
neighbor features for every node simultaneously. This O(|E|) operation is the
computational backbone of every GNN — everything else is normalization and
learnable transformations on top of this primitive.
"""


<a id="2"></a>
## 2. The Message Passing Neural Network (MPNN) Framework

Gilmer et al. (2017) unified all GNN architectures under a single framework: **Message Passing**.

At each layer $l$, every node $v$ updates its representation $h_v^{(l)}$ via three operations:

### Step 1: Message Construction
$$m_{u \to v}^{(l)} = \text{MSG}^{(l)}\left(h_u^{(l-1)}, h_v^{(l-1)}, e_{uv}\right)$$
Each neighbor $u$ of node $v$ constructs a "message" — a transformation of $u$'s features, optionally conditioned on the edge $e_{uv}$.

### Step 2: Aggregation
$$\bar{m}_v^{(l)} = \text{AGG}^{(l)}\left(\{m_{u \to v}^{(l)} : u \in \mathcal{N}(v)\}\right)$$
All incoming messages are combined via a **permutation-invariant** function (sum, mean, or max). Permutation invariance is essential because graphs have no canonical node ordering.

### Step 3: Update
$$h_v^{(l)} = \text{UPDATE}^{(l)}\left(h_v^{(l-1)}, \bar{m}_v^{(l)}\right)$$
The node combines the aggregated messages with its own previous state.

### The Receptive Field
After $K$ layers of message passing, each node's representation encodes information from its **$K$-hop neighborhood**. This is exactly analogous to a CNN's receptive field growing with depth.

| Architecture | MSG Function | AGG Function | UPDATE Function |
|---|---|---|---|
| **GCN** | $W^{(l)} h_u$ | Normalized Sum | Replace |
| **GraphSAGE** | $W^{(l)} h_u$ | Mean / LSTM / Pool | Concat + Transform |
| **GAT** | $W^{(l)} h_u$ | Attention-Weighted Sum | Replace |

📈 **Production Signal:** The MPNN framework is the design pattern. When you read a new GNN paper, the first question to ask is: *"What are the MSG, AGG, and UPDATE functions?"* Every architecture is just a different instantiation of this template.

---


In [ ]:
class ManualMessagePassing:
    """From-scratch MPNN to demystify the framework."""
    
    def __init__(self, in_features, out_features):
        # Learnable weight matrix
        self.W = torch.randn(in_features, out_features) * 0.1
    
    def forward(self, X, A):
        """
        X: Node features [N, F_in]
        A: Adjacency matrix [N, N]
        """
        N = X.shape[0]
        
        # Step 1: MESSAGE — Transform each node's features
        messages = X @ self.W  # [N, F_out]
        
        # Step 2: AGGREGATE — Sum messages from neighbors (A @ messages)
        # Normalize by degree to prevent feature scale explosion
        D_inv = torch.diag(1.0 / A.sum(dim=1).clamp(min=1))
        aggregated = D_inv @ A @ messages  # Mean aggregation
        
        # Step 3: UPDATE — Apply non-linearity
        output = torch.relu(aggregated)
        
        return output

# Test on our small graph
mpnn = ManualMessagePassing(in_features=3, out_features=4)
out = mpnn.forward(X, A)
print(f"Input shape:  {X.shape}  (5 nodes, 3 features)")
print(f"Output shape: {out.shape} (5 nodes, 4 features)")
print(f"\nNode 0 output (aggregated from neighbors 1, 2):\n{out[0]}")

"""
What just happened?
We built a complete message passing layer in ~15 lines. The core operation
is D_inv @ A @ (X @ W) — degree-normalized adjacency times transformed features.
This is the computational DNA of GCN, GraphSAGE, and GAT.
"""


<a id="3"></a>
## 3. Graph Convolutional Networks (GCN) — Kipf & Welling, 2017

### Motivation: From Spectral Theory to a Simple Formula

The original spectral graph convolution involves computing the eigendecomposition of the Laplacian $L = U \Lambda U^T$ and applying learnable filters in the spectral domain. This requires $O(N^3)$ computation — completely impractical for large graphs.

Kipf & Welling showed that by using a first-order Chebyshev polynomial approximation, the entire spectral convolution simplifies to:

### The GCN Layer Rule
$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)$$

Where:
- $\tilde{A} = A + I_N$ — the adjacency matrix with **self-loops** added (so each node includes its OWN features in the aggregation).
- $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$ — the degree matrix of $\tilde{A}$.
- $\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$ — **symmetric normalization** that normalizes by both the sender's AND receiver's degree.
- $W^{(l)} \in \mathbb{R}^{F_{in} \times F_{out}}$ — the learnable weight matrix.

### Why Symmetric Normalization?
Without normalization, $AX$ sums neighbor features — high-degree nodes get huge values, low-degree nodes get tiny ones. Simple row normalization $D^{-1}AX$ (mean) fixes the receiver side but creates asymmetry. Symmetric normalization $D^{-1/2}AD^{-1/2}$ normalizes by BOTH the sender's and receiver's degree, preserving spectral properties and ensuring stable training.

### 🎓 Socratic Deep Check
> *"Why do we add self-loops ($A + I$) before the convolution? What information would be lost without them?"*

<details><summary>👉 Answer</summary>
Without self-loops, the aggregation $AX$ only collects neighbor features — the node's OWN features are excluded. After the update, the node 'forgets' itself entirely. Adding $I$ to $A$ ensures that a node's own representation is included in the aggregation, acting as a residual connection within the graph convolution.
</details>

### Limitation: GCN is Transductive
GCN requires the full adjacency matrix $\tilde{A}$ at both training and inference time. If a new node appears (e.g., a new user joins the social network), you must recompute the entire normalization. This makes GCN impractical for evolving graphs — motivating GraphSAGE.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# GCN Layer — From Scratch (Pure PyTorch)
# ============================================================
class GCNLayerScratch(nn.Module):
    """Implements: sigma(D_tilde^{-1/2} A_tilde D_tilde^{-1/2} H W)"""
    
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Parameter(torch.empty(in_features, out_features))
        nn.init.xavier_uniform_(self.W)
    
    def forward(self, H, A):
        # Step 1: Add self-loops
        A_tilde = A + torch.eye(A.size(0), device=A.device)
        
        # Step 2: Compute symmetric normalization D^{-1/2}
        D_tilde = A_tilde.sum(dim=1)  # Degree vector
        D_inv_sqrt = torch.diag(D_tilde.pow(-0.5))
        
        # Step 3: Normalized adjacency
        A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
        
        # Step 4: Graph convolution
        return torch.relu(A_hat @ H @ self.W)

# Quick test on our 5-node graph
gcn_layer = GCNLayerScratch(3, 4)
out = gcn_layer(X, A)
print(f"GCN from scratch — Output shape: {out.shape}")
print(f"Node 0 embedding: {out[0].detach().numpy().round(3)}")

"""
What just happened?
We implemented the exact GCN propagation rule. The key insight: A_hat @ H
aggregates NORMALIZED neighbor features (including self), then H @ W projects
them into a new feature space. This is a 'convolution' on an irregular graph.
"""


In [ ]:
# ============================================================
# GCN Layer — PyTorch Geometric (Production)
# ============================================================
# PyTorch Geometric implements the same math but uses sparse
# message-passing for scalability to millions of nodes.

try:
    from torch_geometric.nn import GCNConv
    from torch_geometric.data import Data

    # Create a PyG Data object from our graph
    data = Data(x=X, edge_index=edge_index)

    # PyG GCN layer — identical math, sparse implementation
    pyg_gcn = GCNConv(in_channels=3, out_channels=4)
    out_pyg = pyg_gcn(data.x, data.edge_index)
    print(f"PyG GCNConv — Output shape: {out_pyg.shape}")
    print("✅ PyG GCNConv works on sparse edge_index, not dense A matrix.")
    print("   This scales from 5 nodes to 5 million nodes.")
    HAS_PYG = True
except ImportError:
    print("⚠️  PyTorch Geometric not installed. Install with:")
    print("    pip install torch-geometric")
    print("    Continuing with from-scratch implementations.")
    HAS_PYG = False


<a id="4"></a>
## 4. GraphSAGE — Inductive Representation Learning (Hamilton et al., 2017)

### The Problem with GCN
GCN is **transductive**: it operates on the full, fixed graph. It cannot generate embeddings for nodes that were not present during training. In production, this is a dealbreaker — new users join, new molecules are synthesized, new transactions appear.

### The GraphSAGE Innovation
GraphSAGE learns a **generalizable aggregation function** instead of fixed node embeddings. It can generate representations for completely unseen nodes by sampling and aggregating their local neighborhood.

### The Algorithm
For each node $v$ at layer $l$:

**1. Sample** a fixed-size subset of neighbors: $\mathcal{N}_S(v) \subseteq \mathcal{N}(v)$

**2. Aggregate** sampled neighbor features:
$$h_{\mathcal{N}(v)}^{(l)} = \text{AGG}^{(l)}\left(\{h_u^{(l-1)} : u \in \mathcal{N}_S(v)\}\right)$$

**3. Concatenate** with the node's own features and transform:
$$h_v^{(l)} = \sigma\left(W^{(l)} \cdot \text{CONCAT}\left(h_v^{(l-1)},\; h_{\mathcal{N}(v)}^{(l)}\right)\right)$$

**4. Normalize:** $h_v^{(l)} = \frac{h_v^{(l)}}{\|h_v^{(l)}\|_2}$

### Key Differences from GCN
| Aspect | GCN | GraphSAGE |
|---|---|---|
| **Aggregation** | Full neighborhood | Sampled subset |
| **Self-features** | Mixed via $\tilde{A}$ | Explicit CONCAT |
| **Inference** | Requires full graph | Local neighborhood only |
| **Scalability** | $O(N^2)$ dense | $O(K \cdot S^K)$ per node |

### Why CONCAT Instead of SUM?
GCN mixes self-features and neighbor features before the linear transform. GraphSAGE keeps them separate via concatenation, giving the weight matrix $W$ the ability to learn DIFFERENT transformations for "what I know" vs. "what my neighbors know." This explicit separation preserves more signal.

📈 **Production Signal:** Pinterest's **PinSage** (built on GraphSAGE) processes a graph of 3 billion nodes and 18 billion edges. Neighbor sampling makes this tractable — you never need the full graph in memory.


In [ ]:
# ============================================================
# GraphSAGE Layer — From Scratch
# ============================================================
class GraphSAGELayerScratch(nn.Module):
    """Implements: sigma(W * CONCAT(h_v, AGG(h_neighbors)))"""
    
    def __init__(self, in_features, out_features, sample_size=3):
        super().__init__()
        # Note: input is CONCAT of self + neighbor, so 2 * in_features
        self.W = nn.Linear(2 * in_features, out_features, bias=True)
        self.sample_size = sample_size
    
    def _sample_neighbors(self, node_idx, A):
        """Sample a fixed number of neighbors for a node."""
        neighbors = A[node_idx].nonzero(as_tuple=True)[0]
        if len(neighbors) == 0:
            return torch.tensor([node_idx])  # Self-loop fallback
        if len(neighbors) > self.sample_size:
            perm = torch.randperm(len(neighbors))[:self.sample_size]
            neighbors = neighbors[perm]
        return neighbors
    
    def forward(self, H, A):
        N = H.shape[0]
        outputs = []
        
        for v in range(N):
            # Step 1: SAMPLE neighbors
            sampled = self._sample_neighbors(v, A)
            
            # Step 2: AGGREGATE (mean) neighbor features
            h_neighbors = H[sampled].mean(dim=0)  # Mean aggregator
            
            # Step 3: CONCAT self + aggregated neighbors
            h_concat = torch.cat([H[v], h_neighbors])
            
            outputs.append(h_concat)
        
        # Step 4: Transform + activate
        H_concat = torch.stack(outputs)
        out = torch.relu(self.W(H_concat))
        
        # Step 5: L2 Normalize
        out = F.normalize(out, p=2, dim=1)
        return out

# Test
sage_layer = GraphSAGELayerScratch(3, 4, sample_size=2)
out_sage = sage_layer(X, A)
print(f"GraphSAGE from scratch — Output shape: {out_sage.shape}")
print(f"L2 norm of Node 0 embedding: {torch.norm(out_sage[0]):.4f} (should be ~1.0)")

"""
What just happened?
Unlike GCN which needs the FULL adjacency matrix, GraphSAGE only samples
a few neighbors per node. At 2 layers with sample_size=10, each node
touches at most 10 * 10 = 100 nodes — regardless of graph size.
This is how PinSage runs on a 3-billion-node graph.
"""


In [ ]:
# ============================================================
# GraphSAGE — PyTorch Geometric (Production)
# ============================================================
try:
    from torch_geometric.nn import SAGEConv

    pyg_sage = SAGEConv(in_channels=3, out_channels=4)
    out_pyg_sage = pyg_sage(X, edge_index)
    print(f"PyG SAGEConv — Output shape: {out_pyg_sage.shape}")
    print("✅ SAGEConv handles neighbor sampling internally via PyG's DataLoader.")
except ImportError:
    print("⚠️  Skipping PyG SAGEConv (torch-geometric not installed).")


<a id="5"></a>
## 5. Graph Attention Networks (GAT) — Veličković et al., 2018

### The Problem with Fixed Aggregation
Both GCN and GraphSAGE treat all neighbors equally (after degree normalization). But intuitively, not all neighbors are equally relevant. In a citation network, a highly relevant paper should contribute MORE to a node's embedding than a tangentially related one.

### The GAT Innovation: Learned Attention Coefficients
GAT introduces **learnable attention weights** that allow each node to dynamically decide how much "attention" to pay to each neighbor.

### The Mathematics

**Step 1: Linear Transformation**
$$h_i' = W h_i \quad \forall i \in \mathcal{V}$$

**Step 2: Compute Attention Coefficients**
For each edge $(i, j)$, compute a raw attention score using a learnable vector $\vec{a}$:
$$e_{ij} = \text{LeakyReLU}\left(\vec{a}^T \cdot \text{CONCAT}(h_i', h_j')\right)$$

**Step 3: Normalize via Softmax**
$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

**Step 4: Attention-Weighted Aggregation**
$$h_i^{\text{out}} = \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij} \cdot h_j'\right)$$

### Multi-Head Attention
Just like Transformers, GAT uses $K$ independent attention heads and concatenates (or averages) their outputs:
$$h_i^{\text{out}} = \overset{K}{\underset{k=1}{\big\|}} \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}^{(k)} W^{(k)} h_j\right)$$

### 🎓 Socratic Deep Check
> *"The attention vector $\vec{a}$ has shape $[2F']$, and it's the SAME for all edges. How can a single shared vector produce DIFFERENT attention scores for different node pairs?"*

<details><summary>👉 Answer</summary>
The vector $\vec{a}$ is shared, but the INPUT to it ($\text{CONCAT}(h_i', h_j')$) is different for every edge. The attention mechanism learns a linear scoring function: which PATTERNS of source-target feature combinations indicate high relevance. The diversity comes from the node features, not from the attention parameters.
</details>

📈 **Production Signal:** GATs are the architecture of choice when neighbor importance is non-uniform — fraud detection (suspicious vs. normal connections), knowledge graphs (different relation types), and heterogeneous graphs.


In [ ]:
# ============================================================
# GAT Layer — From Scratch
# ============================================================
class GATLayerScratch(nn.Module):
    """Single-head Graph Attention Layer."""
    
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        # Attention vector: takes CONCAT of two transformed features
        self.a = nn.Linear(2 * out_features, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)
    
    def forward(self, H, A):
        N = H.shape[0]
        
        # Step 1: Linear transformation
        H_prime = self.W(H)  # [N, F_out]
        
        # Step 2: Compute attention for ALL edges
        # For each pair (i,j), compute a^T * CONCAT(h_i', h_j')
        # Efficient: expand H_prime for all pairs
        H_i = H_prime.unsqueeze(1).repeat(1, N, 1)  # [N, N, F_out]
        H_j = H_prime.unsqueeze(0).repeat(N, 1, 1)  # [N, N, F_out]
        concat = torch.cat([H_i, H_j], dim=-1)      # [N, N, 2*F_out]
        
        e = self.leaky_relu(self.a(concat)).squeeze(-1)  # [N, N]
        
        # Step 3: Mask non-edges and softmax normalize
        # Add self-loops for attention
        A_with_self = A + torch.eye(N, device=A.device)
        e = e.masked_fill(A_with_self == 0, float('-inf'))
        alpha = F.softmax(e, dim=1)  # [N, N] attention weights
        
        # Step 4: Weighted aggregation
        out = alpha @ H_prime  # [N, F_out]
        
        return torch.elu(out), alpha

# Test
gat_layer = GATLayerScratch(3, 4)
out_gat, attention_weights = gat_layer(X, A)
print(f"GAT from scratch — Output shape: {out_gat.shape}")
print(f"\nAttention weights for Node 2 (neighbors: 0, 1, 3, self):")
print(f"  α(2→0) = {attention_weights[2, 0]:.4f}")
print(f"  α(2→1) = {attention_weights[2, 1]:.4f}")
print(f"  α(2→2) = {attention_weights[2, 2]:.4f} (self-attention)")
print(f"  α(2→3) = {attention_weights[2, 3]:.4f}")
print(f"  Sum    = {attention_weights[2, :].sum():.4f} (should be 1.0)")

"""
What just happened?
Unlike GCN's fixed normalization (1/sqrt(deg_i * deg_j)), GAT LEARNS which
neighbors matter most. The attention weights alpha are differentiable —
the model discovers that some connections are more informative than others.
Note how the weights sum to 1.0 (softmax normalization over neighbors).
"""


In [ ]:
# ============================================================
# Multi-Head GAT — From Scratch
# ============================================================
class MultiHeadGATScratch(nn.Module):
    def __init__(self, in_features, out_features, num_heads=4):
        super().__init__()
        self.heads = nn.ModuleList([
            GATLayerScratch(in_features, out_features) for _ in range(num_heads)
        ])
    
    def forward(self, H, A):
        head_outputs = [head(H, A)[0] for head in self.heads]
        # Concatenate all heads: [N, num_heads * out_features]
        return torch.cat(head_outputs, dim=-1)

mh_gat = MultiHeadGATScratch(3, 4, num_heads=4)
out_mh = mh_gat(X, A)
print(f"Multi-Head GAT — Output shape: {out_mh.shape}  (4 heads × 4 features = 16)")

# ============================================================
# GAT — PyTorch Geometric (Production)
# ============================================================
try:
    from torch_geometric.nn import GATConv

    pyg_gat = GATConv(in_channels=3, out_channels=4, heads=4, concat=True)
    out_pyg_gat = pyg_gat(X, edge_index)
    print(f"PyG GATConv — Output shape: {out_pyg_gat.shape}")
    print("✅ PyG GATConv uses sparse attention — scales to massive graphs.")
except ImportError:
    print("⚠️  Skipping PyG GATConv (torch-geometric not installed).")


<a id="6"></a>
## 6. Application — Molecular Property Prediction (Node Classification)

### The Problem
Given a molecule represented as a graph (atoms = nodes, bonds = edges), predict a property of each atom or of the entire molecule. This is a core task in computational chemistry and drug discovery.

We'll use the **Cora citation dataset** — a benchmark where:
- **Nodes** = academic papers (2,708)
- **Edges** = citation links (5,429)
- **Node features** = bag-of-words vectors (1,433 dimensions)
- **Task** = classify each paper into one of 7 research topics

This is **semi-supervised node classification**: only a small subset of nodes have labels, and the model must propagate label information through the graph structure.

📈 **Production Signal:** The same architecture applied to Cora powers fraud detection (classify transaction nodes as fraud/legit), drug discovery (classify atoms by chemical properties), and social network analysis (classify users by interest).

---


In [ ]:
# ============================================================
# COMPLETE GNN PIPELINE: Node Classification on Cora
# ============================================================
try:
    from torch_geometric.datasets import Planetoid
    from torch_geometric.nn import GCNConv, GATConv
    import torch.optim as optim

    # 1. Load the Cora dataset
    dataset = Planetoid(root='/tmp/Cora', name='Cora')
    data = dataset[0]
    print(f"Dataset: {dataset}")
    print(f"  Nodes:    {data.num_nodes}")
    print(f"  Edges:    {data.num_edges}")
    print(f"  Features: {data.num_node_features}")
    print(f"  Classes:  {dataset.num_classes}")
    print(f"  Train:    {data.train_mask.sum().item()} nodes")
    print(f"  Val:      {data.val_mask.sum().item()} nodes")
    print(f"  Test:     {data.test_mask.sum().item()} nodes")

    # 2. Define a 2-layer GCN
    class GCN(nn.Module):
        def __init__(self, num_features, num_classes, hidden_dim=64):
            super().__init__()
            self.conv1 = GCNConv(num_features, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, num_classes)
            self.dropout = nn.Dropout(0.5)
        
        def forward(self, data):
            x, edge_index = data.x, data.edge_index
            x = self.conv1(x, edge_index)
            x = F.relu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            return F.log_softmax(x, dim=1)

    # 3. Define a 2-layer GAT (for comparison)
    class GAT(nn.Module):
        def __init__(self, num_features, num_classes, hidden_dim=8, heads=8):
            super().__init__()
            self.conv1 = GATConv(num_features, hidden_dim, heads=heads, dropout=0.6)
            self.conv2 = GATConv(hidden_dim * heads, num_classes, heads=1,
                                 concat=False, dropout=0.6)
            self.dropout = nn.Dropout(0.6)
        
        def forward(self, data):
            x, edge_index = data.x, data.edge_index
            x = self.dropout(x)
            x = self.conv1(x, edge_index)
            x = F.elu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            return F.log_softmax(x, dim=1)

    # 4. Training Function
    def train_and_evaluate(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        history = {'train_loss': [], 'val_acc': [], 'test_acc': []}
        best_val_acc = 0
        best_test_acc = 0
        
        for epoch in range(epochs):
            # Train
            model.train()
            optimizer.zero_grad()
            out = model(data)
            loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()
            history['train_loss'].append(loss.item())
            
            # Evaluate
            model.eval()
            with torch.no_grad():
                pred = model(data).argmax(dim=1)
                val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
                test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
            history['val_acc'].append(val_acc)
            history['test_acc'].append(test_acc)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_test_acc = test_acc
            
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1:3d} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")
        
        return history, best_val_acc, best_test_acc

    # 5. Train both models
    print("\n" + "="*60)
    print("Training GCN...")
    print("="*60)
    gcn_model = GCN(dataset.num_node_features, dataset.num_classes)
    gcn_hist, gcn_val, gcn_test = train_and_evaluate(gcn_model, data)

    print(f"\n" + "="*60)
    print("Training GAT...")
    print("="*60)
    gat_model = GAT(dataset.num_node_features, dataset.num_classes)
    gat_hist, gat_val, gat_test = train_and_evaluate(gat_model, data, lr=0.005)

    print(f"\n" + "="*60)
    print("RESULTS COMPARISON")
    print("="*60)
    print(f"  GCN  — Best Val Acc: {gcn_val:.4f} | Test Acc: {gcn_test:.4f}")
    print(f"  GAT  — Best Val Acc: {gat_val:.4f} | Test Acc: {gat_test:.4f}")

except ImportError:
    print("⚠️  PyTorch Geometric is required for this section.")
    print("    Install: pip install torch-geometric")
    print("    The from-scratch implementations above demonstrate the core concepts.")


In [ ]:
# ============================================================
# VISUALIZATION: Training Curves
# ============================================================
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].plot(gcn_hist['train_loss'], label='GCN', color='#2196F3', lw=2)
    axes[0].plot(gat_hist['train_loss'], label='GAT', color='#FF5722', lw=2)
    axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('NLL Loss')
    axes[0].legend(fontsize=12)
    axes[0].grid(alpha=0.3)

    # Accuracy curves
    axes[1].plot(gcn_hist['test_acc'], label='GCN', color='#2196F3', lw=2)
    axes[1].plot(gat_hist['test_acc'], label='GAT', color='#FF5722', lw=2)
    axes[1].set_title('Test Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(fontsize=12)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    
    print("\n📈 Note: GAT typically achieves slightly higher accuracy than GCN")
    print("   because it learns WHICH neighbors matter, rather than treating all equally.")

except NameError:
    print("⚠️  Skipping visualization (requires training results from previous cell).")


---
## ✅ Knowledge Check

### Q1: What does multiplying $AX$ compute, and why is it the foundation of all GNNs?
<details><summary>👉 Answer</summary>
Row $i$ of $AX$ equals the sum of feature vectors of all neighbors of node $i$. This single matrix multiplication IS neighborhood aggregation — the core primitive of message passing. Every GNN architecture (GCN, GraphSAGE, GAT) is a variation of this operation with different normalization and weighting schemes.
</details>

### Q2: Why does GCN use symmetric normalization $\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$ instead of simple row normalization $D^{-1}A$?
<details><summary>👉 Answer</summary>
Simple row normalization ($D^{-1}A$, mean aggregation) normalizes by the receiver's degree only. A message from a high-degree hub node carries the same weight as one from a low-degree leaf. Symmetric normalization scales by BOTH the sender's and receiver's degrees ($1/\sqrt{d_i \cdot d_j}$), dampening the influence of hub nodes and preserving the spectral properties of the graph Laplacian — critical for stable training.
</details>

### Q3: Why is GraphSAGE called "inductive" while GCN is "transductive"?
<details><summary>👉 Answer</summary>
GCN computes embeddings using the full, fixed adjacency matrix. Adding a new node requires recomputing the entire normalization. GraphSAGE learns a generalizable AGGREGATION FUNCTION — given any node and its local neighborhood, it can generate an embedding. This means it can embed completely unseen nodes at inference time, making it suitable for dynamic, evolving graphs.
</details>

### Q4: How does GAT determine the importance of each neighbor?
<details><summary>👉 Answer</summary>
GAT uses a learnable attention mechanism: for each edge $(i,j)$, it computes $e_{ij} = \text{LeakyReLU}(\vec{a}^T [W h_i \| W h_j])$, then normalizes across all neighbors via softmax to get $\alpha_{ij}$. The shared attention vector $\vec{a}$ learns which feature patterns between source-target pairs indicate high relevance. This is analogous to Transformer self-attention but constrained to graph neighborhoods.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. **Verify the AX trick:** Create a 4-node graph with known features. Manually compute $AX$ on paper, then verify with code. Confirm that row $i$ of $AX$ is the sum of node $i$'s neighbor features.
2. **Self-loop effect:** Compute node embeddings with and without self-loops ($A$ vs $A + I$). Explain why the output differs and which is more useful for node classification.

### Tier 2: Intermediate
1. **Over-smoothing experiment:** Stack 2, 4, 8, and 16 GCN layers on Cora and plot the test accuracy vs. depth. Observe the over-smoothing collapse and compute the pairwise cosine similarity of node embeddings at each depth.
2. **Aggregation ablation:** Modify the from-scratch GraphSAGE to use MAX pooling instead of MEAN aggregation. Compare node classification accuracy and explain the theoretical trade-off (mean preserves distribution statistics; max captures salient features).

### Tier 3: Advanced
1. **Attention visualization:** Extract GAT attention weights on Cora and visualize them as a heatmap for a local subgraph. Identify which citation links receive the highest attention — do they correspond to papers in the same class?
2. **Molecular property prediction:** Load a molecular dataset (e.g., `MoleculeNet` via PyG) where atoms are nodes and bonds are edges. Train a 3-layer GAT to predict molecular toxicity. Compare against a baseline that ignores graph structure (MLP on global fingerprints) to quantify the structural advantage of GNNs.
